# Single Symbol Verify

单票验证示例：展示单个标的的最小回放与撮合比对流程。

`加载数据 -> prepare_market_data -> process_workflow -> compare_df`



In [3]:
import logging
import sys
from pathlib import Path

project_root = Path.cwd()
if (
    not (project_root / "examples").exists()
    and (project_root.parent / "examples").exists()
):
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from examples.helper import (  # noqa: E402
    DEFAULT_CUT_TIME,
    build_legacy_loader,
    load_symbol_frames,
    run_single_validation,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s.%(msecs)03d | %(levelname)-7s | %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)

DATE = "2026-04-02"
BROKER = "guoxin"
COLO = "SZ"
SYMBOL = "600000"  # 你要测哪只票就填哪只
MARKET = "SH"  # 600000 是 SH；如果测 000/300 开头就改成 "SZ"
QUOTE_DIR = "/root/ZW_HDS/quote_data_intern/extracted/guoxin/SZ"
BASE_ARCHIVE_DIR = "/mnt/beegfs/quant002/hds_work/DailyMd/"
CUT_TIME = DEFAULT_CUT_TIME


loader = build_legacy_loader(
    date=DATE,
    broker=BROKER,
    colo=COLO,
    base_archive_dir=BASE_ARCHIVE_DIR,
)

In [4]:
trade_df, order_df = load_symbol_frames(loader, QUOTE_DIR, SYMBOL)
matched, order_data, orderbook = run_single_validation(
    trade_df=trade_df,
    order_df=order_df,
    symbol=SYMBOL,
    market=MARKET,
    cut_time=CUT_TIME,
)

print(f"single check: {'OK' if matched else 'MISMATCH'}")
order_data.head()

single check: OK


,serial,type_serial,mi_type,local_time,exchange_time,channel,symbol,market,int_time,trade_price,...,trade_amount,sell_id,buy_id,biz_index,type,order_type,order_index,order_volume,orderorino,order_price
0,3569682,2748211,257,1775093101075962005,91500050,5,600000,49,91500050,NaN,...,NaN,NaN,NaN,4664,O,A,4664.0,100.0,205.0,102400.0
1,3569683,2748212,257,1775093101075963481,91500090,5,600000,49,91500090,NaN,...,NaN,NaN,NaN,4665,O,A,4665.0,100.0,927.0,92200.0
2,3569684,2748213,257,1775093101075965524,91500090,5,600000,49,91500090,NaN,...,NaN,NaN,NaN,4666,O,A,4666.0,100.0,1002.0,92200.0
3,3569685,2748214,257,1775093101075967965,91500090,5,600000,49,91500090,NaN,...,NaN,NaN,NaN,4667,O,A,4667.0,5000.0,1139.0,102100.0
4,3569686,2748215,257,1775093101075970202,91500090,5,600000,49,91500090,NaN,...,NaN,NaN,NaN,4668,O,A,4668.0,5000.0,1150.0,101100.0


In [ ]:
orderbook.export_trades_csv("trades.csv")
orderbook.export_cancels_csv("cancels.csv")

批次的随机抽取股票验证
直接加载上面的配置进行抽取


In [9]:
import random
import time

import polars as pl

STOCK_TYPES = {0, 1, 2, 4}
SAMPLE_SH = 100
SAMPLE_SZ = 100
SEED = 42

date_str = DATE.replace("-", "")
quote_dir_path = Path(QUOTE_DIR)
baseinfo_candidates = [
    quote_dir_path / date_str / "baseinfo.txt",
    quote_dir_path / date_str / "baseinfo.csv",
    quote_dir_path / "baseinfo.txt",
    quote_dir_path / "baseinfo.csv",
]
baseinfo_path = next((path for path in baseinfo_candidates if path.exists()), None)
if baseinfo_path is None:
    raise FileNotFoundError(
        "找不到 baseinfo 文件，已检查: "
        + ", ".join(str(path) for path in baseinfo_candidates)
    )

baseinfo = pl.read_csv(
    baseinfo_path,
    schema_overrides={"ticker": pl.Utf8},
    infer_schema_length=10000,
)

stock_codes = (
    baseinfo.filter(pl.col("security_type").is_in(list(STOCK_TYPES)))
    .get_column("ticker")
    .cast(pl.Utf8)
    .str.zfill(6)
    .to_list()
)

sh_codes = [code for code in stock_codes if code.startswith("6")]
sz_codes = [code for code in stock_codes if code.startswith(("0", "3"))]

random.seed(SEED)
sh_sample = random.sample(sh_codes, min(SAMPLE_SH, len(sh_codes)))
random.seed(SEED)
sz_sample = random.sample(sz_codes, min(SAMPLE_SZ, len(sz_codes)))
batch_symbols = [(symbol, "SH") for symbol in sh_sample] + [
    (symbol, "SZ") for symbol in sz_sample
]

print(f"baseinfo 路径: {baseinfo_path}")
print(f"沪市可用: {len(sh_codes)}, 抽取: {len(sh_sample)}")
print(f"深市可用: {len(sz_codes)}, 抽取: {len(sz_sample)}")

print("正在加载全量 transaction 数据...")
t0 = time.perf_counter()
all_trade_df = loader.load(
    instruments="all",
    quote_type="transaction",
    quote_dir=QUOTE_DIR,
    evaluate_eagerly=True,
).with_columns(pl.col("symbol").cast(pl.Utf8).str.slice(0, 6).alias("symbol"))
print(f"  transaction: {all_trade_df.shape}, 耗时 {time.perf_counter() - t0:.1f}s")

print("正在加载全量 order 数据...")
t0 = time.perf_counter()
all_order_df = loader.load(
    instruments="all",
    quote_type="order",
    quote_dir=QUOTE_DIR,
    evaluate_eagerly=True,
).with_columns(pl.col("symbol").cast(pl.Utf8).str.slice(0, 6).alias("symbol"))
print(f"  order: {all_order_df.shape}, 耗时 {time.perf_counter() - t0:.1f}s")

baseinfo 路径: /root/ZW_HDS/quote_data_intern/extracted/guoxin/SZ/20260402/baseinfo.txt
沪市可用: 2308, 抽取: 100
深市可用: 2886, 抽取: 100
正在加载全量 transaction 数据...
  transaction: (187921048, 18), 耗时 148.3s
正在加载全量 order 数据...
  order: (261978603, 16), 耗时 174.5s


In [10]:
SAMPLE_SH = 100
SAMPLE_SZ = 100

In [11]:
results = []
failed = []


def run_one(symbol, market, all_trade, all_order):
    """Run symbol-level validation against the preloaded order and trade data."""
    t0 = time.perf_counter()
    trade_sym = all_trade.filter(pl.col("symbol") == symbol)
    order_sym = all_order.filter(pl.col("symbol") == symbol)
    if trade_sym.is_empty() and order_sym.is_empty():
        msg = "无数据: symbol=" + symbol
        raise ValueError(msg)
    matched, order_data, _orderbook = run_single_validation(
        trade_df=trade_sym,
        order_df=order_sym,
        symbol=symbol,
        market=market,
        cut_time=CUT_TIME,
    )
    elapsed = time.perf_counter() - t0
    return matched, elapsed, len(order_data)


for i, (sym, market) in enumerate(batch_symbols, start=1):
    try:
        match, elapsed, rows = run_one(sym, market, all_trade_df, all_order_df)
        results.append(
            {
                "symbol": sym,
                "market": market,
                "match": match,
                "rows": rows,
                "time_s": round(elapsed, 2),
            }
        )
        status = "OK" if match else "MISMATCH"
        print(f"[{i}/{len(batch_symbols)}] {market} {sym}: {status} ({elapsed:.1f}s)")
    except Exception as e:
        failed.append({"symbol": sym, "market": market, "error": str(e)})
        print(f"[{i}/{len(batch_symbols)}] {market} {sym}: ERROR - {e}")

[1/200] SH 600496: OK (2.5s)
[2/200] SH 600368: OK (0.7s)
[3/200] SH 688511: OK (0.4s)
[4/200] SH 600330: OK (9.9s)
[5/200] SH 603386: OK (0.9s)
[6/200] SH 688370: OK (0.4s)
[7/200] SH 688369: OK (0.6s)
[8/200] SH 600843: OK (0.5s)
[9/200] SH 600586: OK (2.3s)
[10/200] SH 600113: OK (1.4s)
[11/200] SH 603637: OK (0.6s)
[12/200] SH 600018: OK (5.9s)
[13/200] SH 600127: OK (3.3s)
[14/200] SH 688012: OK (5.4s)
[15/200] SH 601033: OK (0.9s)
[16/200] SH 603639: OK (0.9s)
[17/200] SH 600753: OK (0.3s)
[18/200] SH 600739: OK (1.1s)
[19/200] SH 600481: OK (3.4s)
[20/200] SH 603173: OK (0.7s)
[21/200] SH 603689: OK (0.7s)
[22/200] SH 601006: OK (11.6s)
[23/200] SH 601137: OK (1.4s)
[24/200] SH 688818: OK (2.5s)
[25/200] SH 600917: OK (0.8s)
[26/200] SH 601117: OK (4.1s)
[27/200] SH 605377: OK (0.6s)
[28/200] SH 603259: OK (11.8s)
[29/200] SH 601211: OK (3.6s)
[30/200] SH 603298: OK (0.8s)
[31/200] SH 600377: OK (1.3s)
[32/200] SH 603281: OK (0.5s)
[33/200] SH 603102: OK (0.4s)
[34/200] SH 68837

16:37:44.622 | ERROR   | pylob.matching_engine - [001257] 限价单价格必须大于0，当前价格: -1728647296
16:37:44.642 | WARNING | pylob.matching_engine - [001257] 无法找到订单 ID: 21315705


[195/200] SZ 001257: OK (14.8s)
[196/200] SZ 002564: OK (1.0s)
[197/200] SZ 002226: OK (1.7s)
[198/200] SZ 000532: OK (0.8s)
[199/200] SZ 002587: OK (2.4s)
[200/200] SZ 002434: OK (1.6s)


In [ ]:
results_df = pl.DataFrame(results)
failed_df = pl.DataFrame(failed) if failed else pl.DataFrame()

total = len(results)
matched = sum(1 for item in results if item["match"])
mismatched = total - matched

print()
print("=" * 50)
print(f"总计完成: {total}, 匹配: {matched}, 不匹配: {mismatched}, 失败: {len(failed)}")
if total:
    print(f"总匹配率: {matched}/{total}")

if not results_df.is_empty():
    print()
    print(results_df.sort(["market", "symbol"]).head(10))

if not failed_df.is_empty():
    print()
    print(failed_df.head(10))